## Imports

In [21]:
from pathlib import Path
import json
import random
import time
from copy import deepcopy

import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

## raíz del proyecto y device

In [22]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("No pude encontrar la raíz del proyecto.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
WORKING_DIR = DATA_DIR / "working"
MANIFESTS_DIR = DATA_DIR / "manifests"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "isic_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_SIZE = 300
BATCH_SIZE = 16
NUM_WORKERS = 4
PIN_MEMORY = True

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

MODEL_NAME = "efficientnet_b3"
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 30

USE_MIXED_PRECISION = True
EARLY_STOPPING_PATIENCE = 6
EARLY_STOPPING_MIN_DELTA = 5e-4

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PROJECT_ROOT: /mnt/d/Universidad/analitica/proyecto_analitica2
OUTPUT_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/outputs/isic_baseline
DEVICE: cuda
GPU: NVIDIA GeForce RTX 4060


## cargar subset y label map

In [23]:
subset_path = WORKING_DIR / "isic" / "subsets" / "isic_subset_large.csv"
subset_df = pd.read_csv(subset_path)

# usar el texto real de la clase presente en el subset
if "target_text" in subset_df.columns:
    subset_df["target_text"] = subset_df["target_text"].astype(str)
elif "target_label" in subset_df.columns and not pd.api.types.is_numeric_dtype(subset_df["target_label"]):
    subset_df["target_text"] = subset_df["target_label"].astype(str)
else:
    raise ValueError("No encontré una columna de texto de clase como 'target_text' o 'target_label' string.")

# reconstruir mapping SOLO con clases presentes en este subset
labels_present = sorted(subset_df["target_text"].unique().tolist())
label_to_idx = {label: idx for idx, label in enumerate(labels_present)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

subset_df["target_idx"] = subset_df["target_text"].map(label_to_idx).astype(int)

train_df = subset_df[subset_df["split_final"] == "train"].copy().reset_index(drop=True)
val_df = subset_df[subset_df["split_final"] == "val"].copy().reset_index(drop=True)
test_df = subset_df[subset_df["split_final"] == "test"].copy().reset_index(drop=True)

print("subset_df:", subset_df.shape)
print("train_df:", train_df.shape)
print("val_df:", val_df.shape)
print("test_df:", test_df.shape)

print("\nClases presentes en el subset:")
print(labels_present)

print("\nlabel_to_idx:")
print(label_to_idx)

display(subset_df.head())

subset_df: (13000, 25)
train_df: (10000, 25)
val_df: (1500, 25)
test_df: (1500, 25)

Clases presentes en el subset:
['AK', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'SCC', 'VASC']

label_to_idx:
{'AK': 0, 'BCC': 1, 'BKL': 2, 'DF': 3, 'MEL': 4, 'NV': 5, 'SCC': 6, 'VASC': 7}


,image_name,file_path,target_label,labels_list,label_count,age_approx,anatom_site_general,lesion_id,sex,dataset_name,...,AK,BKL,DF,VASC,SCC,UNK,group_id,split_final,target_text,target_idx
0,ISIC_0067679,/mnt/d/Universidad/analitica/proyecto_analitic...,BCC,['BCC'],1,60.0,anterior torso,BCN_0001624,male,ISIC_2019,...,0.0,0.0,0.0,0.0,0.0,0.0,BCN_0001624,train,BCC,1
1,ISIC_0033597,/mnt/d/Universidad/analitica/proyecto_analitic...,NV,['NV'],1,NaN,NaN,HAM_0005439,NaN,ISIC_2019,...,0.0,0.0,0.0,0.0,0.0,0.0,HAM_0005439,train,NV,5
2,ISIC_0025825,/mnt/d/Universidad/analitica/proyecto_analitic...,AK,['AK'],1,80.0,head/neck,HAM_0002232,female,ISIC_2019,...,1.0,0.0,0.0,0.0,0.0,0.0,HAM_0002232,train,AK,0
3,ISIC_0010573,/mnt/d/Universidad/analitica/proyecto_analitic...,NV,['NV'],1,30.0,posterior torso,NaN,male,ISIC_2019,...,0.0,0.0,0.0,0.0,0.0,0.0,ISIC_0010573,train,NV,5
4,ISIC_0059979,/mnt/d/Universidad/analitica/proyecto_analitic...,NV,['NV'],1,35.0,anterior torso,BCN_0000789,male,ISIC_2019,...,0.0,0.0,0.0,0.0,0.0,0.0,BCN_0000789,train,NV,5


## separar train / val / test

In [24]:
train_df = subset_df[subset_df["split_final"] == "train"].copy()
val_df = subset_df[subset_df["split_final"] == "val"].copy()
test_df = subset_df[subset_df["split_final"] == "test"].copy()

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (10000, 25)
val: (1500, 25)
test: (1500, 25)


## transforms

In [25]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.75, 1.00),
        ratio=(0.90, 1.10)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(25),
    transforms.ColorJitter(
        brightness=0.20,
        contrast=0.20,
        saturation=0.15,
        hue=0.03
    ),
    transforms.ToTensor(),
    transforms.RandomErasing(
        p=0.25,
        scale=(0.02, 0.12),
        ratio=(0.3, 3.3),
        value="random"
    ),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print(train_transform)
print(eval_transform)

Compose(
    RandomResizedCrop(size=(300, 300), scale=(0.75, 1.0), ratio=(0.9, 1.1), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomVerticalFlip(p=0.3)
    RandomRotation(degrees=[-25.0, 25.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.85, 1.15), hue=(-0.03, 0.03))
    ToTensor()
    RandomErasing(p=0.25, scale=(0.02, 0.12), ratio=(0.3, 3.3), value=random, inplace=False)
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
Compose(
    Resize(size=(300, 300), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


## dataset

In [26]:
class ISICDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = Path(row["file_path"])
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        target = int(row["target_idx"])

        return {
            "image": image,
            "target": torch.tensor(target, dtype=torch.long),
            "image_name": row["image_name"],
            "file_path": str(image_path),
            "target_idx": target,
            "target_text": str(row["target_text"]),
            "split_final": row["split_final"],
        }

## class weights y sampler

In [27]:
NUM_CLASSES = len(label_to_idx)

train_class_counts = (
    train_df["target_idx"]
    .value_counts()
    .reindex(range(NUM_CLASSES), fill_value=0)
    .sort_index()
)

class_count_df = pd.DataFrame({
    "target_idx": range(NUM_CLASSES),
    "target_text": [idx_to_label[i] for i in range(NUM_CLASSES)],
    "count": train_class_counts.values
})

print("Distribución en train:")
display(class_count_df)

# sampler por frecuencia inversa
sample_weights = train_df["target_idx"].map(
    lambda idx: 1.0 / train_class_counts.loc[idx]
).astype(np.float32).values

sample_weights = sample_weights / sample_weights.mean()
sample_weights = np.clip(sample_weights, 0.5, 5.0)

sample_weights_tensor = torch.tensor(sample_weights, dtype=torch.double)

print("Resumen sample_weights:")
print("min:", sample_weights.min())
print("mean:", sample_weights.mean())
print("max:", sample_weights.max())

Distribución en train:


,target_idx,target_text,count
0,0,AK,380
1,1,BCC,1314
2,2,BKL,1079
3,3,DF,92
4,4,MEL,1732
5,5,NV,5066
6,6,SCC,240
7,7,VASC,97


Resumen sample_weights:
min: 0.5
mean: 0.9678
max: 5.0


In [28]:
train_dataset = ISICDataset(train_df, transform=train_transform)
val_dataset = ISICDataset(val_df, transform=eval_transform)
test_dataset = ISICDataset(test_df, transform=eval_transform)

print("Tamaños de datasets:")
print("train:", len(train_dataset))
print("val:", len(val_dataset))
print("test:", len(test_dataset))

Tamaños de datasets:
train: 10000
val: 1500
test: 1500


## dataloaders

In [29]:
train_sampler = WeightedRandomSampler(
    weights=sample_weights_tensor,
    num_samples=len(sample_weights_tensor),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0)
)

print("Dataloaders listos.")
print("train batches:", len(train_loader))
print("val batches:", len(val_loader))
print("test batches:", len(test_loader))

Dataloaders listos.
train batches: 625
val batches: 94
test batches: 94


## modelo preentrenado

In [30]:
NUM_CLASSES = len(label_to_idx)

weights = models.EfficientNet_B3_Weights.DEFAULT
model = models.efficientnet_b3(weights=weights)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(DEVICE)

print(model.classifier)
print("NUM_CLASSES:", NUM_CLASSES)

Sequential(
  (0): Dropout(p=0.3, inplace=True)
  (1): Linear(in_features=1536, out_features=8, bias=True)
)
NUM_CLASSES: 8


In [31]:
print("Clases en train:", sorted(train_df["target_idx"].unique().tolist()))
print("Clases en val:", sorted(val_df["target_idx"].unique().tolist()))
print("Clases en test:", sorted(test_df["target_idx"].unique().tolist()))
print("Clases del label_map:", list(range(len(label_to_idx))))

Clases en train: [0, 1, 2, 3, 4, 5, 6, 7]
Clases en val: [0, 1, 2, 3, 4, 5, 6, 7]
Clases en test: [0, 1, 2, 3, 4, 5, 6, 7]
Clases del label_map: [0, 1, 2, 3, 4, 5, 6, 7]


## congelar backbone al inicio

In [32]:
print("No se congelará el backbone en esta versión.")
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("Parámetros entrenables:", trainable_params)
print("Parámetros totales:", total_params)

No se congelará el backbone en esta versión.
Parámetros entrenables: 10708528
Parámetros totales: 10708528


## loss, optimizer y scheduler

In [33]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.05
)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda" and USE_MIXED_PRECISION))

print("criterion:", criterion)

criterion: CrossEntropyLoss()


/tmp/ipykernel_62056/3933797265.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda" and USE_MIXED_PRECISION))


## función de evaluación

In [34]:
def run_one_epoch(model, loader, criterion, optimizer=None, scaler=None, device="cpu"):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    all_preds = []
    all_targets = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        targets = batch["target"].to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast("cuda", enabled=(device == "cuda" and USE_MIXED_PRECISION)):
                logits = model(images)
                loss = criterion(logits, targets)

            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * images.size(0)

        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_targets.extend(targets.detach().cpu().numpy().tolist())

    epoch_loss = total_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    epoch_f1_macro = f1_score(all_targets, all_preds, average="macro")
    epoch_f1_micro = f1_score(all_targets, all_preds, average="micro")

    return {
        "loss": epoch_loss,
        "acc": epoch_acc,
        "f1_macro": epoch_f1_macro,
        "f1_micro": epoch_f1_micro,
        "preds": all_preds,
        "targets": all_targets,
    }

## función de entrenamiento

In [35]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best_score = None
        self.counter = 0
        self.should_stop = False

    def step(self, current_score):
        if self.best_score is None:
            self.best_score = current_score
            return True

        improved = False
        if self.mode == "max":
            improved = current_score > (self.best_score + self.min_delta)
        else:
            improved = current_score < (self.best_score - self.min_delta)

        if improved:
            self.best_score = current_score
            self.counter = 0
            return True
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
            return False

##  preparar unfreeze

In [36]:
print("No se usará unfreeze_all en esta versión.")

No se usará unfreeze_all en esta versión.


## entrenamiento en dos fases con early stopping

In [37]:
history = []
best_model_state = None
best_epoch = -1
best_val_f1_macro = -np.inf

early_stopper = EarlyStopping(
    patience=EARLY_STOPPING_PATIENCE,
    min_delta=EARLY_STOPPING_MIN_DELTA,
    mode="max"
)

start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()

    train_metrics = run_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=DEVICE
    )

    val_metrics = run_one_epoch(
        model=model,
        loader=val_loader,
        criterion=criterion,
        optimizer=None,
        scaler=None,
        device=DEVICE
    )

    scheduler.step(val_metrics["f1_macro"])

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "train_f1_macro": train_metrics["f1_macro"],
        "train_f1_micro": train_metrics["f1_micro"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
        "val_f1_macro": val_metrics["f1_macro"],
        "val_f1_micro": val_metrics["f1_micro"],
        "lr": optimizer.param_groups[0]["lr"],
        "epoch_time_sec": time.time() - t0,
    }
    history.append(row)

    improved = early_stopper.step(val_metrics["f1_macro"])
    if improved:
        best_val_f1_macro = val_metrics["f1_macro"]
        best_epoch = epoch
        best_model_state = deepcopy(model.state_dict())

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"train_f1_macro={train_metrics['f1_macro']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_f1_macro={val_metrics['f1_macro']:.4f} | "
        f"lr={optimizer.param_groups[0]['lr']:.2e}"
    )

    if early_stopper.should_stop:
        print(f"\n⏹️ Early stopping activado en epoch {epoch}. Mejor epoch: {best_epoch}")
        break

total_time = time.time() - start_time
print(f"\nTiempo total de entrenamiento: {total_time/60:.2f} min")
print(f"Mejor epoch: {best_epoch}")
print(f"Mejor val_f1_macro: {best_val_f1_macro:.4f}")

Epoch 01/30 | train_loss=1.3490 | train_f1_macro=0.5185 | val_loss=1.0044 | val_f1_macro=0.5521 | lr=1.00e-04
Epoch 02/30 | train_loss=0.9362 | train_f1_macro=0.7328 | val_loss=0.9313 | val_f1_macro=0.5574 | lr=1.00e-04
Epoch 03/30 | train_loss=0.8005 | train_f1_macro=0.7985 | val_loss=0.9200 | val_f1_macro=0.5692 | lr=1.00e-04
Epoch 04/30 | train_loss=0.7057 | train_f1_macro=0.8385 | val_loss=0.9322 | val_f1_macro=0.5764 | lr=1.00e-04
Epoch 05/30 | train_loss=0.6440 | train_f1_macro=0.8678 | val_loss=0.9553 | val_f1_macro=0.5551 | lr=1.00e-04
Epoch 06/30 | train_loss=0.5945 | train_f1_macro=0.8885 | val_loss=0.9374 | val_f1_macro=0.5867 | lr=1.00e-04
Epoch 07/30 | train_loss=0.5542 | train_f1_macro=0.9034 | val_loss=0.9513 | val_f1_macro=0.5903 | lr=1.00e-04
Epoch 08/30 | train_loss=0.5344 | train_f1_macro=0.9079 | val_loss=0.9653 | val_f1_macro=0.5854 | lr=1.00e-04
Epoch 09/30 | train_loss=0.4978 | train_f1_macro=0.9277 | val_loss=0.9587 | val_f1_macro=0.6109 | lr=1.00e-04
Epoch 10/3

In [38]:
if best_model_state is None:
    raise ValueError("No se guardó ningún mejor modelo.")

model.load_state_dict(best_model_state)

test_metrics = run_one_epoch(
    model=model,
    loader=test_loader,
    criterion=criterion,
    optimizer=None,
    scaler=None,
    device=DEVICE
)

print("Resultados en test:")
print(f"test_loss     = {test_metrics['loss']:.4f}")
print(f"test_acc      = {test_metrics['acc']:.4f}")
print(f"test_f1_macro = {test_metrics['f1_macro']:.4f}")
print(f"test_f1_micro = {test_metrics['f1_micro']:.4f}")

Resultados en test:
test_loss     = 0.9324
test_acc      = 0.7327
test_f1_macro = 0.6133
test_f1_micro = 0.7327


## guardar historial y checkpoint

In [39]:
history_df = pd.DataFrame(history)

history_path = OUTPUT_DIR / "isic_baseline_history.csv"
checkpoint_path = OUTPUT_DIR / "isic_baseline_efficientnet_b3.pt"
metrics_path = OUTPUT_DIR / "isic_test_metrics.json"

history_df.to_csv(history_path, index=False)

torch.save({
    "model_state_dict": best_model_state,
    "label_to_idx": label_to_idx,
    "image_size": IMAGE_SIZE,
    "num_classes": NUM_CLASSES,
    "best_val_f1_macro": float(best_val_f1_macro),
    "best_epoch": int(best_epoch),
    "test_loss": float(test_metrics["loss"]),
    "test_acc": float(test_metrics["acc"]),
    "test_f1_macro": float(test_metrics["f1_macro"]),
    "test_f1_micro": float(test_metrics["f1_micro"]),
}, checkpoint_path)

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump({
        "best_val_f1_macro": float(best_val_f1_macro),
        "best_epoch": int(best_epoch),
        "test_loss": float(test_metrics["loss"]),
        "test_acc": float(test_metrics["acc"]),
        "test_f1_macro": float(test_metrics["f1_macro"]),
        "test_f1_micro": float(test_metrics["f1_micro"]),
    }, f, ensure_ascii=False, indent=2)

print("Guardado en:")
print("-", history_path)
print("-", checkpoint_path)
print("-", metrics_path)

Guardado en:
- /mnt/d/Universidad/analitica/proyecto_analitica2/outputs/isic_baseline/isic_baseline_history.csv
- /mnt/d/Universidad/analitica/proyecto_analitica2/outputs/isic_baseline/isic_baseline_efficientnet_b3.pt
- /mnt/d/Universidad/analitica/proyecto_analitica2/outputs/isic_baseline/isic_test_metrics.json


history_df

In [40]:
labels_order = list(range(len(idx_to_label)))
target_names = [idx_to_label[i] for i in labels_order]

print(classification_report(
    test_metrics["targets"],
    test_metrics["preds"],
    labels=labels_order,
    target_names=target_names,
    zero_division=0
))

cm = confusion_matrix(test_metrics["targets"], test_metrics["preds"], labels=labels_order)
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
display(cm_df)

              precision    recall  f1-score   support

          AK       0.49      0.57      0.53        44
         BCC       0.73      0.68      0.70       195
         BKL       0.56      0.50      0.53       169
          DF       0.62      0.50      0.56        10
         MEL       0.62      0.58      0.60       263
          NV       0.82      0.88      0.85       776
         SCC       0.50      0.31      0.38        32
        VASC       0.80      0.73      0.76        11

    accuracy                           0.73      1500
   macro avg       0.64      0.59      0.61      1500
weighted avg       0.73      0.73      0.73      1500



,AK,BCC,BKL,DF,MEL,NV,SCC,VASC
AK,25,6,5,0,2,5,1,0
BCC,8,132,16,0,15,19,5,0
BKL,11,13,85,0,22,37,1,0
DF,0,1,0,5,0,4,0,0
MEL,3,9,14,0,152,80,3,2
NV,0,13,26,2,53,682,0,0
SCC,4,7,6,1,1,3,10,0
VASC,0,0,0,0,0,3,0,8
